# Part II — implicit time-stepping, encoding initial/boundary conditions, open problems

Companion notebook to the Part II slides (`slides/part2.pdf`) and to Section *Part II* of the
lecture notes. It follows the slides in order:

1. **Recap: is an explicit scheme enough?** (CFL bound, implicit Euler) — slides 3–5
2. **Solving $Au=b$ within the MPS representation** (why not direct, why not Krylov) — slides 6–9
3. **DMRG in a slide** (ALS, two-site DMRG, AMEn) — slides 10–12
4. **Back to the heat equation** (warm starts, Crank–Nicolson) — slide 13
5. **Which bond dimension? And when to stop?** (error budget, grid refinement) — slide 14
6. **Encoding boundary/initial conditions** (analytic, TT-SVD, TT-cross) — slides 15–16
7. **Matrix cross interpolation** (pivots, maxvol) — slide 17
8. **Extension to TTs: TT-cross** (and how it fails) — slides 18–20
9. **TT-cross for encoding geometry** (smoothed masks) — slides 21–22
10. **Open problems** — slide 23

Exercises are interleaved where the concept is introduced. Each exercise is followed
directly by a cell marked **Solution**: try it first, predict, then measure.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import qtade_quimb as qq
import qtade_tn as tn

plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.grid": True, "grid.alpha": 0.3,
                     "font.size": 10})

## 1. Recap: is an explicit scheme enough?

Part I represented functions as QTTs and stepped the heat equation explicitly,

$$\partial_t u = \alpha\,\partial_x^2 u \quad\Longrightarrow\quad u^{k+1} = (\mathbb{1} + \alpha\Delta t\,L)\,u^k ,$$

with $L$ the second-difference MPO. Two primitives: apply an operator, then round.
Explicit Euler is only stable under the Courant (CFL) condition

$$\frac{\alpha\,\Delta t}{\Delta x^2} \le \frac{1}{2}.$$

Watch it fail. We seed the initial condition with a $10^{-8}$ sawtooth — a stand-in for
round-off — and step just below and just above the bound.

In [ ]:
n, alpha = 8, 1.0
N = 2 ** n
h = 1.0 / (N + 1)                     # Dirichlet grid: x_i = (i + 1) h
xs = (np.arange(N) + 1) * h
L = tn.qtt_laplacian(n, dx=h)
roundoff = 1e-8 * (-1.0) ** np.arange(N)  # highest-frequency mode, rank 1

for ratio in (0.4, 0.6):
    dt = ratio * h ** 2 / alpha
    E = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, alpha * dt)), 1e-13)
    u = tn.qtt_from_vector(np.sin(np.pi * xs) + roundoff, eps=1e-14)
    for _ in range(200):
        u = tn.tt_round(tn.mpo_apply(E, u), eps=1e-12)
    print(f"alpha dt/dx^2 = {ratio}: max|u| after 200 steps = "
          f"{np.abs(tn.qtt_to_vector(u)).max():.2e}")

Above the bound the sawtooth mode is amplified by $|1 - 4\cdot 0.6| = 1.4$ per step and
swamps the solution. Below it, nothing happens. Now look at what the bound costs as
quantics buys resolution: $\Delta x = 2^{-n}$ forces $\Delta t \sim 4^{-n}$. Memory grows
like $n$; the number of steps grows like $4^n$.

In [ ]:
print(f"{'n':>3} {'N':>12} {'memory ~ n*chi^2':>18} {'steps to t = 0.1':>20}")
for n_ in (10, 15, 20, 25, 30):
    dt = 0.5 * (2.0 ** -n_) ** 2
    print(f"{n_:>3} {2**n_:>12,d} {n_ * 100:>18,d} {int(0.1 / dt):>20,d}")

At $n=20$ you can *store* the field on a laptop and would still need $10^{11}$ steps.
That is the motivation of the slides in two lines:

* **Why implicit solvers?** Stability.
* **Why stability?** To decouple the time discretisation from the space discretisation.

Implicit (backward) Euler,

$$(\mathbb{1} - \alpha\Delta t\,L)\,u^{k+1} = u^k ,$$

is A-stable: the amplification factor $(1+\alpha\Delta t\lambda)^{-1}$, $\lambda\ge0$ an
eigenvalue of $-L$, never exceeds one. $\Delta t$ now affects only accuracy, not
stability — at the price of **a linear solve every time step**.

## 2. Solving $Au=b$ within the MPS representation

Classical solvers come in three families:

| family | idea | in TT format |
|---|---|---|
| direct | operate on matrix elements until the matrix is easy to invert | ✗ no easy access to matrix elements* |
| iterative | improve a guess with global operations (CG, GMRES) | ✓ TT-GMRES, Riemannian solvers |
| variational | vary the parameters of the solution to minimise a loss | ✓ TT-ALS, TT-mALS, TT-AMEn |

\* Unless you are lucky: the QTT Laplacian has an explicit low-rank inverse (Kazeev &
Khoromskij 2012).

Ground rules: $A$ (an MPO) and $b$ (an MPS) are low rank; we may not know the rank of the
solution; the ideal solver finds the solution *and* its rank. Useful fact: MPS of fixed
ranks form a smooth manifold.

**Why not Krylov?** Each matrix–vector product multiplies the rank by $\chi_{\rm MPO}$ and
each vector addition adds ranks. Even with rounding after every operation, the
*intermediate* iterates can be far more complex than the answer. Here is conjugate
gradients written entirely in TT arithmetic.

In [ ]:
def tt_cg(A, b, iters, eps=1e-10):
    # Conjugate gradients with a rounding step after every TT operation.
    x = tn.tt_scale(b, 0.0)
    r, p = b, b
    rr = tn.tt_dot(r, r)
    hist = []
    for k in range(iters):
        Ap = tn.tt_round(tn.mpo_apply(A, p), eps=eps)
        a = rr / tn.tt_dot(p, Ap)
        x = tn.tt_round(tn.tt_add(x, tn.tt_scale(p, a)), eps=eps)
        r = tn.tt_round(tn.tt_add(r, tn.tt_scale(Ap, -a)), eps=eps)
        rr_new = tn.tt_dot(r, r)
        p = tn.tt_round(tn.tt_add(r, tn.tt_scale(p, rr_new / rr)), eps=eps)
        rr = rr_new
        hist.append((k + 1, max(tn.tt_ranks(x)), max(tn.tt_ranks(r)),
                     np.sqrt(rr) / tn.tt_norm(b)))
    return x, hist


n = 10
N = 2 ** n
h = 1.0 / (N + 1)
xs = (np.arange(N) + 1) * h
L = tn.qtt_laplacian(n, dx=h)
A = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, -1e-4)), 1e-13)
b = tn.qtt_from_vector(np.exp(-((xs - 0.5) / 0.05) ** 2), eps=1e-12)

x_cg, hist = tt_cg(A, b, 40)
print(f"{'iter':>5} {'chi(x_k)':>9} {'chi(r_k)':>9} {'|r_k|/|b|':>11}")
for k, cx, cr, res in hist[::5]:
    print(f"{k:>5} {cx:>9} {cr:>9} {res:>11.2e}")
x_ref = tn.dmrg_solve(A, b, sweeps=4, eps=1e-10, chi_max=64)
print(f"\nrank of the converged solution (DMRG): {max(tn.tt_ranks(x_ref))}")

The iterates and residuals carry two to three times the rank of the solution, and the
residual stalls where truncation starts destroying the Krylov orthogonality. Riemannian
methods avoid the rank blow-up by staying on the fixed-rank manifold, but need retraction
and vector-transport routines and are not rank adaptive by construction. For linear
problems the variational route is usually the better deal.

## 3. DMRG in a slide

Reformulate the linear system as a variational problem,

$$\min_x \|Ax-b\|^2 \qquad\text{or}\qquad \min_x \tfrac12 x^\top A x - x^\top b \quad (A\succ 0).$$

A TN is multilinear, so freeze all tensors but one: the reduced problem is a small dense
linear system. Solve it, move on, sweep back and forth, and use the gauge freedom (mixed
canonical form) to keep the local problems well conditioned. Three variants:

| | TT-ALS | TT-mALS (two-site DMRG) | TT-AMEn |
|---|---|---|---|
| update | single site | two sites | single site |
| rank | fixed | adaptive (SVD of the pair) | adaptive (residual enrichment) |
| caveat | sensitive to the initial guess | local solves are expensive | — guarantees beyond SPD |

`qtade_tn` implements ALS (`als_solve`) and two-site DMRG (`dmrg_solve`). Backward Euler
with a time step $10^5$ times the explicit limit:

In [ ]:
n, alpha = 14, 1.0
h = 2.0 ** -n
dt = 1e-5                            # 10^5 times the explicit limit
xs = np.linspace(0, 1, 2 ** n, endpoint=False)

L = tn.qtt_laplacian(n, dx=h)
A = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, -alpha * dt)), 1e-13)
u0 = tn.qtt_from_vector(np.exp(-((xs - 0.5) / 0.05) ** 2), eps=1e-10)

print(f"explicit limit dt = {0.5 * h**2:.2e};  we are using dt = {dt:.0e}")
print(f"A: MPO rank {max(tn.mpo_ranks(A))}, built analytically — no assembly step")
print(f"b: MPS rank {max(tn.tt_ranks(u0))}\n")

print("two-site DMRG, cold start:")
x_dmrg = tn.dmrg_solve(A, u0, sweeps=4, eps=1e-10, chi_max=64, verbose=True)

### One-site ALS: the same idea with rank adaptivity removed

ALS freezes all cores but one and solves exactly. It cannot change $\chi$. Watch what
that costs when the fixed rank is too small.

In [ ]:
for chi in (4, 8, 16, 32):
    guess = tn.tt_round([c.copy() for c in u0], eps=1e-14, chi_max=chi)
    xa = tn.als_solve(A, u0, x0=guess, sweeps=6)
    print(f"ALS at fixed chi = {chi:>2}: residual {tn.residual(A, xa, u0):.2e}")
print(f"\ntwo-site DMRG chose chi = {max(tn.tt_ranks(x_dmrg))} by itself, "
      f"residual {tn.residual(A, x_dmrg, u0):.2e}")

The one-site residual plateaus at a level set by the rank you guessed, and no number of
sweeps improves it. That single limitation is why two-site DMRG and AMEn exist: the
truncated SVD inside a two-site update *chooses* the rank from a tolerance, and AMEn gets
the same effect at one-site cost by enriching each core with residual directions.

## 4. Back to the heat equation

On the slides, backward Euler + DMRG reaches a target error of $10^{-2}$ up to $89\times$
faster than explicit Euler at $n=12$, because it takes far fewer steps. The main practical
lever is the **warm start**: consecutive time steps differ by $O(\Delta t)$, so the previous
solution is an excellent initial guess.

In [ ]:
u = u0
cold_t, warm_t = [], []
for k in range(3):
    t0 = time.perf_counter()
    _ = tn.dmrg_solve(A, u, sweeps=2, eps=1e-10, chi_max=64)
    cold_t.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    u = tn.dmrg_solve(A, u, x0=u, sweeps=1, eps=1e-10, chi_max=64)
    warm_t.append(time.perf_counter() - t0)

print(f"cold start, 2 sweeps: {np.mean(cold_t):.3f} s/step")
print(f"warm start, 1 sweep : {np.mean(warm_t):.3f} s/step  "
      f"({np.mean(cold_t) / np.mean(warm_t):.1f}x faster)")

Compare schemes by **wall-clock time to a target error**, never by cost per step:
implicit stepping loses on the second measure and wins on the first.

---
### Exercise 1 — Crank–Nicolson, and what second order buys

The solver is a component you swap the scheme around, not the scheme itself. Backward
Euler is first order in time. Crank–Nicolson,

$$\left(\mathbb{1} - \tfrac{\alpha\Delta t}{2}L\right)u^{k+1}
 = \left(\mathbb{1} + \tfrac{\alpha\Delta t}{2}L\right)u^{k},$$

is second order and still unconditionally stable. Build both, integrate to $T = 0.005$,
plot the error against $\Delta t$ on log axes and read off the slopes (expect 1 and 2).
Then answer: **why is the cost per step essentially unchanged?**

In [ ]:
n = 12
N1 = 2 ** n
h = 2.0 ** -n
L = tn.qtt_laplacian(n, dx=h)
T = 0.005

# Compare against the *semi-discrete* exact solution, not the continuous one. The
# discrete Dirichlet Laplacian has known eigenpairs, so this reference is exact in space
# and the only error left is the time discretisation -- which is the thing being
# measured. Comparing against the continuous solution instead puts a spatial error floor
# under the plot and hides the second-order convergence completely.
j = 1
v0 = np.sin(np.pi * j * (np.arange(N1) + 1) / (N1 + 1))
lam = -4.0 / h ** 2 * np.sin(np.pi * j / (2 * (N1 + 1))) ** 2
exact = lambda t: v0 * np.exp(lam * t)


def integrate_backward_euler(dt):
    A = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, -dt)), 1e-13)
    u = tn.qtt_from_vector(exact(0.0), eps=1e-13)
    for _ in range(int(round(T / dt))):
        u = tn.dmrg_solve(A, u, x0=u, sweeps=2, eps=1e-12, chi_max=40)
    return u


def integrate_crank_nicolson(dt):
    # TODO: build A (left-hand side) and B (right-hand side) as MPOs, then loop:
    #   rhs = round(mpo_apply(B, u));  u = dmrg_solve(A, rhs, x0=u, ...)
    raise NotImplementedError


# for dt in (T / 2, T / 4, T / 8, T / 16):
#     err_be = ...
#     err_cn = ...
#     print(dt, err_be, err_cn)

#### Solution — Exercise 1

In [ ]:
def integrate_cn(dt):
    A = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, -dt / 2)), 1e-13)
    B = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, dt / 2)), 1e-13)
    u = tn.qtt_from_vector(exact(0.0), eps=1e-13)
    for _ in range(int(round(T / dt))):
        rhs = tn.tt_round(tn.mpo_apply(B, u), eps=1e-12)
        u = tn.dmrg_solve(A, rhs, x0=u, sweeps=2, eps=1e-12, chi_max=40)
    return u


ref = exact(T)
dts, e_be, e_cn = [], [], []
for dt in (T / 2, T / 4, T / 8, T / 16):
    dts.append(dt)
    e_be.append(np.linalg.norm(tn.qtt_to_vector(integrate_backward_euler(dt)) - ref)
                / np.linalg.norm(ref))
    e_cn.append(np.linalg.norm(tn.qtt_to_vector(integrate_cn(dt)) - ref)
                / np.linalg.norm(ref))
dts = np.array(dts)
print(f"backward Euler slope: {np.polyfit(np.log(dts), np.log(e_be), 1)[0]:.2f}")
print(f"Crank-Nicolson slope: {np.polyfit(np.log(dts), np.log(e_cn), 1)[0]:.2f}")
plt.loglog(dts, e_be, "o-", label="backward Euler")
plt.loglog(dts, e_cn, "s-", label="Crank-Nicolson")
plt.xlabel("$\\Delta t$"), plt.ylabel("relative error"), plt.legend()
plt.tight_layout()
plt.show()
print("Cost per step is unchanged because the extra work is one MPO application on the")
print("right-hand side -- O(n chi^2 chi_MPO^2) -- against a linear solve that costs")
print("O(n chi^4) or worse. The solve dominates either way.")

## 5. Which bond dimension? And when to stop?

Three error sources now compete:

1. **discretisation** error (space $O(\Delta x^2)$ and time $O(\Delta t)$),
2. **representation** error (truncation, set by $\varepsilon$ and $\chi_{\max}$),
3. **solver** error (the residual).

Rule of thumb: make sure discretisation error dominates. To see the solver term we
degrade the solver on purpose by capping its bond dimension.

In [ ]:
# One Fourier mode decays as exp(-alpha k^2 t): a known exact solution.
n = 12
h = 2.0 ** -n
xs = np.linspace(0, 1, 2 ** n, endpoint=False)
kmode = 2 * np.pi
L = tn.qtt_laplacian(n, dx=h)
exact_mode = lambda t: np.sin(kmode * xs) * np.exp(-kmode ** 2 * t)

T = 0.005
print(f"{'dt':>10} {'solver chi_max':>15} {'solver resid':>14} {'error vs exact':>16}")
for dt in (T / 4, T / 64):
    A = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, -dt)), 1e-13)
    for chi_cap in (2, 4, 40):
        u = tn.qtt_from_vector(exact_mode(0.0), eps=1e-12)
        last = 0.0
        for _ in range(int(round(T / dt))):
            prev = u
            u = tn.dmrg_solve(A, prev, x0=prev, sweeps=1, eps=1e-12, chi_max=chi_cap)
            last = tn.residual(A, u, prev)
        err = (np.linalg.norm(tn.qtt_to_vector(u) - exact_mode(T))
               / np.linalg.norm(exact_mode(T)))
        print(f"{dt:>10.2e} {chi_cap:>15} {last:>14.2e} {err:>16.2e}")

Read the table in blocks of three. At the coarse $\Delta t$, tightening the solver from
$\chi=4$ to $\chi=40$ changes nothing: time discretisation dominates and every extra sweep
is wasted. At the fine $\Delta t$ the sloppy solve is suddenly what limits you. Target a
residual about an order of magnitude below the expected discretisation error, and no
tighter.

**A low residual does not mean a low error**, and the real test of a numerical solution is
that it **converges under grid refinement**. Solve $-u'' = \pi^2\sin(\pi x)$ with
$u(0)=u(1)=0$ (exact solution $\sin\pi x$) on ever finer quantics grids:

In [ ]:
print(f"{'n':>3} {'N':>7} {'residual':>10} {'max error':>11} {'error ratio':>12}")
prev_err = None
for n_ in (4, 6, 8, 10, 12, 14):
    N_ = 2 ** n_
    h_ = 1.0 / (N_ + 1)
    x_ = (np.arange(N_) + 1) * h_
    A_ = tn.mpo_scale(tn.qtt_laplacian(n_, dx=h_), -1.0)      # -d^2/dx^2, SPD
    f_ = tn.qtt_from_vector(np.pi ** 2 * np.sin(np.pi * x_), eps=1e-14)
    u_ = tn.dmrg_solve(A_, f_, sweeps=4, eps=1e-12, chi_max=32)
    err = np.abs(tn.qtt_to_vector(u_) - np.sin(np.pi * x_)).max()
    ratio = "" if prev_err is None else f"{prev_err / err:12.1f}"
    print(f"{n_:>3} {N_:>7,d} {tn.residual(A_, u_, f_):>10.1e} {err:>11.2e} {ratio}")
    prev_err = err

Every solve has a tiny residual, yet at $n=4$ the error is $3\times10^{-3}$: the residual
certifies the *solve*, not the *solution* — a certificate of merit, not of authenticity.
Each refinement by $n\to n+2$ should cut the error by $16$ (second order). It does, until
the conditioning ($\sim 4^n$) amplifies the solver residual and the error stops improving
at $n=14$: from there on, refinement buys nothing unless the solver tolerance follows.

---
### Exercise 2 — two-dimensional Poisson, and the rank you actually need

The pressure solve of Part III, in miniature. Solve $\nabla^2 p = f$ on a $2^m\times2^m$
grid with a smooth right-hand side, using `qq.laplacian_2d` and `tn.dmrg_solve`. Then:

* sweep `chi_max` from 4 to 64 and record the residual — which bond dimension do you need?
* time the solve at $m = 5, 6, 7, 8$ and check whether the cost grows like $m$ (i.e.
  logarithmically in the number of cells) or like $4^m$.

In [ ]:
m = 6
N = 2 ** m
xx = np.linspace(0, 1, N, endpoint=False)
X, Y = np.meshgrid(xx, xx, indexing="ij")
rhs_dense = np.sin(2 * np.pi * X) * np.sin(2 * np.pi * Y)

# TODO (a): b = qq.from_grid(rhs_dense, ...) ; Lap = qq.laplacian_2d(m, h=1.0 / N)
# TODO (b): for chi in (4, 8, 16, 32, 64): solve and record tn.residual(...)
# TODO (c): time the chi=32 solve for m in (5, 6, 7, 8) and plot time against m

#### Solution — Exercise 2

In [ ]:
b = qq.from_grid(rhs_dense, eps=1e-12)
Lap = qq.laplacian_2d(m, h=1.0 / N)
print("chi_max sweep at m = 6:")
for chi in (4, 8, 16, 32, 64):
    p = tn.dmrg_solve(Lap, b, sweeps=3, eps=1e-10, chi_max=chi)
    print(f"  chi_max = {chi:>3}: residual {tn.residual(Lap, p, b):.2e}, "
          f"chi used {max(tn.tt_ranks(p))}")

print("\nscaling with resolution at chi_max = 32:")
ms, ts = [], []
for mi in (5, 6, 7, 8):
    Ni = 2 ** mi
    xi = np.linspace(0, 1, Ni, endpoint=False)
    Xi, Yi = np.meshgrid(xi, xi, indexing="ij")
    bi = qq.from_grid(np.sin(2 * np.pi * Xi) * np.sin(2 * np.pi * Yi), eps=1e-12)
    Li = qq.laplacian_2d(mi, h=1.0 / Ni)
    t0 = time.perf_counter()
    tn.dmrg_solve(Li, bi, sweeps=2, eps=1e-10, chi_max=32)
    ts.append(time.perf_counter() - t0)
    ms.append(mi)
    print(f"  m = {mi}: {Ni**2:>9,d} cells, {ts[-1]:.2f} s")
print(f"\ncells grew by {4**(ms[-1]-ms[0]):,d}x, runtime by {ts[-1]/ts[0]:.1f}x")

The $x$-axis of the timing is $m = \log_2 N$. If the runtime is anything close to linear
in $m$, the cost is polylogarithmic in the number of grid cells, which is the entire
claim of the method.

## 6. Encoding boundary/initial conditions

So far we assumed that initial and boundary data arrive already in QTT form. Three routes
to a quantics train:

| route | cost | needs | notes |
|---|---|---|---|
| explicit constructions | free | a closed form (polynomials, sin, cos, exp) | exact |
| TT-SVD | $O(2^n)$ | the function on the entire grid | exact, quasi-optimal; randomised variants exist |
| TT-cross (TCI) | $O(n\chi^2)$ evaluations | only a black box $f$ | adaptive queries; no convergence guarantee, extremely efficient |

The same function, $e^{-2x}$, three ways:

In [ ]:
n = 16
xs = np.arange(2 ** n) / 2 ** n
ref = np.exp(-2 * xs)
w = 2.0 ** -(np.arange(n) + 1)

routes = {
    "explicit (qtt_exp)": tn.qtt_exp(n, rate=-2.0),
    "TT-SVD": tn.qtt_from_vector(ref, eps=1e-12),
    "TT-cross": tn.tt_cross(lambda idx: np.exp(-2 * (idx @ w)), [2] * n, rank=2, sweeps=2),
}
for name, c in routes.items():
    err = np.linalg.norm(tn.qtt_to_vector(c) - ref) / np.linalg.norm(ref)
    print(f"{name:>20}: chi = {max(tn.tt_ranks(c))}, rel err = {err:.1e}")

All three agree. The difference is what they *need*: TT-SVD touched all $2^{16}$ values,
TT-cross touched a few hundred, the analytic construction none. Only TT-cross scales to
functions that have no closed form and too many grid points to tabulate.

## 7. Matrix cross interpolation

Select $r$ columns $\mathcal J$ and rows $\mathcal I$ (the *pivots*) and invert the
intersecting $r\times r$ *pivot matrix*:

$$A = A(\mathbb I, \mathbb J) \approx A(\mathbb I, \mathcal J)\,A(\mathcal I, \mathcal J)^{-1} A(\mathcal I, \mathbb J).$$

If $A$ has exact rank $r$ and the pivot matrix is nonsingular, this is exact. Good pivots
make it *stable*: the **maxvol** heuristic picks the $r\times r$ submatrix of (nearly)
maximal $|\det|$, which bounds the interpolation coefficients by about one.

In [ ]:
rng = np.random.default_rng(0)
Q, _ = np.linalg.qr(rng.standard_normal((400, 8)))
rows = tn.maxvol(Q)
B = Q @ np.linalg.inv(Q[rows])
print(f"maxvol picked rows {np.sort(rows)}")
print(f"max |entry| of Q inv(Q[rows]) = {np.abs(B).max():.3f}   "
      f"(bounded near 1: that bound is why cross interpolation is stable)")

---
### Exercise 3 — cross-interpolate a matrix from a few rows and columns

`entry(I, J)` below returns the submatrix $A(I, J)$ of a $600\times500$ matrix
$A_{ij} = \sin(x_i+y_j) + (x_i-y_j)^2$. Pretend it is expensive.

1. What is the rank of $A$? (Think before computing.)
2. Implement `cross_matrix(entry, shape, r)`: start from $r$ random columns, then
   alternate *rows = maxvol(columns)*, *columns = maxvol(rows)* a few times, and return the
   skeleton reconstruction. Use a QR of the sampled block before `tn.maxvol` for stability.
3. Report the max error and the fraction of entries you evaluated for $r = 3, 4, 5, 6$.

In [ ]:
x_rows = np.linspace(0, 1, 600)
y_cols = np.linspace(0, 2, 500)
_n_entries = {"count": 0}


def entry(I, J):
    I, J = np.atleast_1d(I), np.atleast_1d(J)
    _n_entries["count"] += len(I) * len(J)
    return np.sin(x_rows[I, None] + y_cols[None, J]) + (x_rows[I, None] - y_cols[None, J]) ** 2


def cross_matrix(entry, shape, r, iters=3, seed=0):
    # TODO
    raise NotImplementedError

#### Solution — Exercise 3

In [ ]:
def cross_matrix(entry, shape, r, iters=3, seed=0):
    m, n_ = shape
    all_rows, all_cols = np.arange(m), np.arange(n_)
    J = np.random.default_rng(seed).choice(n_, r, replace=False)
    for _ in range(iters):
        Q, _ = np.linalg.qr(entry(all_rows, J))      # r columns
        I = tn.maxvol(Q)
        Q, _ = np.linalg.qr(entry(I, all_cols).T)    # r rows
        J = tn.maxvol(Q)
    C, R, P = entry(all_rows, J), entry(I, all_cols), entry(I, J)
    return C @ np.linalg.solve(P, R)


full = np.sin(x_rows[:, None] + y_cols[None, :]) + (x_rows[:, None] - y_cols[None, :]) ** 2
for r in (3, 4, 5, 6):
    _n_entries["count"] = 0
    approx = cross_matrix(entry, full.shape, r)
    print(f"r = {r}: max error {np.abs(approx - full).max():.1e}, "
          f"entries evaluated {_n_entries['count']:>7,d} "
          f"({100 * _n_entries['count'] / full.size:.0f}% of A, mostly repeated sweeps)")
print("\nRank 5: sin(x+y) = sin x cos y + cos x sin y (2) plus (x-y)^2 = x^2 - 2xy + y^2 (3).")
print("At r = 5 the skeleton is exact to machine precision. At r = 6 the pivot matrix is")
print("singular in exact arithmetic -- maxvol keeps it usable, but more pivots than the")
print("rank buy nothing. A real implementation stops adding pivots when the error stalls.")

## 8. Extension to TTs: TT-cross

Apply the matrix skeleton recursively: unfold the tensor as (first index) × (the rest),
cross-interpolate, keep the pivot rows as a *nested* set of left multi-indices, and move
on to the next index. The result is a train of "$T$-cores" (sampled fibres) separated by
inverse pivot matrices, which merge into an ordinary MPS. Every core is built from
$O(d\chi^2)$ function evaluations, so the whole train costs $O(nd\chi^2)$ and never forms
the dense tensor.

`tn.tt_cross` is the plain fixed-rank version. It takes a function of an integer
multi-index array and never sees a grid.

In [ ]:
n = 20
weights = 2.0 ** -(np.arange(n) + 1)


def f_black_box(idx):
    # f evaluated at binary multi-indices. Could be a lookup, a solver, anything.
    xv = idx @ weights
    return np.exp(-2 * xv) + np.sin(6 * np.pi * xv)


cores, stats = tn.tt_cross(f_black_box, [2] * n, rank=6, sweeps=3, return_stats=True)
xs = np.linspace(0, 1, 2 ** n, endpoint=False)
exact_v = np.exp(-2 * xs) + np.sin(6 * np.pi * xs)
err = np.linalg.norm(tn.qtt_to_vector(cores) - exact_v) / np.linalg.norm(exact_v)
print(f"grid points:      {2**n:,d}")
print(f"f evaluations:    {stats['evaluations']:,d}  "
      f"({100 * stats['evaluations'] / 2**n:.3f}% of the grid)")
print(f"bond dimensions:  {tn.tt_ranks(cores)}")
print(f"relative error:   {err:.2e}")

Machine precision from a fraction of a percent of the grid. Practical variants
(Núñez Fernández *et al.* 2025) differ in how they choose pivots: **TCI** keeps the pivots
nested for a global interpolation; **rook search** greedily adds pivots after each sweep;
**informed pivots** seed them from known structure of $f$; **DMRG-cross** searches new
pivots on a two-site block; **AMEn-cross** uses AMEn residual directions; **TT-Opt** uses
the cross machinery for grid-search optimisation.

### How TT-cross fails

Cross interpolation is *exact on the fibres it samples*: a virtue for clean data and a
liability for dirty data.

In [ ]:
n = 16
w = 2.0 ** -(np.arange(n) + 1)
xs = np.linspace(0, 1, 2 ** n, endpoint=False)
noise_field = 0.01 * np.random.default_rng(2).standard_normal(2 ** n)

cases = {
    "clean smooth f": (lambda idx: np.exp(-2 * (idx @ w)), np.exp(-2 * xs)),
    "f + 1% noise": (
        lambda idx: np.exp(-2 * (idx @ w)) + noise_field[
            (idx * (2 ** (n - 1 - np.arange(n)))).sum(axis=1)],
        np.exp(-2 * xs) + noise_field),
    "spike, width 1e-3": (
        lambda idx: np.exp(-((idx @ w - 0.371) / 1e-3) ** 2),
        np.exp(-((xs - 0.371) / 1e-3) ** 2)),
    "spike, width 1e-5": (
        lambda idx: np.exp(-((idx @ w - 0.6180339887) / 1e-5) ** 2),
        np.exp(-((xs - 0.6180339887) / 1e-5) ** 2)),
}
for name, (fn, ref) in cases.items():
    for rank in (8, 16):
        c = tn.tt_cross(fn, [2] * n, rank=rank, sweeps=3)
        got = tn.qtt_to_vector(c)
        print(f"{name:>20} rank {rank:>2}: rel err = "
              f"{np.linalg.norm(got - ref) / np.linalg.norm(ref):.2e}   "
              f"max |value found| = {np.abs(got).max():.3f} (true {np.abs(ref).max():.1f})")

Three lessons.

* The noisy case is reproduced **faithfully, noise included**: cross interpolates, it does
  not denoise. Doubling the rank makes the error *worse* — the extra pivots fit noise, so
  under noise TT-cross overestimates the rank. (*Robust TT recovery* is one of the open
  problems on slide 23.)
* The width-$10^{-3}$ spike is found without difficulty.
* The width-$10^{-5}$ spike — about one grid cell wide, at an irrational position — is
  **missed completely**, at every rank. No pivot lands near it, the method never learns it
  exists, and nothing in the output says anything is wrong. If you know where the
  interesting region is, tell the algorithm (informed pivots).

---
### Exercise 4 — cross-interpolate something you cannot see

TT-cross turns "I can evaluate $f$" into "I have $f$ in compressed form", and the
evaluation count is the figure of merit. `mystery` is a black box on $2^{18}$ points: you
may not look inside it, and you may not build the dense array.

1. Reconstruct it with `tn.tt_cross` at ranks 2, 4, 8, 16 and record the number of
   evaluations.
2. Decide, from the rank at which the reconstruction stops improving, roughly how
   complicated the function is. (Measure the error on a few thousand random points,
   not on the full grid.)
3. Only then, plot it.

In [ ]:
_n = 18
_w = 2.0 ** -(np.arange(_n) + 1)
_calls = {"count": 0}


def mystery(idx):
    _calls["count"] += len(idx)
    t = idx @ _w
    return (np.exp(-3 * t) * np.cos(10 * np.pi * t) + 0.3 * np.sin(2 * np.pi * t))


# TODO: reconstruct, count evaluations, compare against 2**_n = 262,144.

#### Solution — Exercise 4

In [ ]:
def tt_eval(cores, idx):
    # Evaluate a train at a batch of multi-indices without forming the dense vector.
    out = np.ones((len(idx), 1))
    for k, c in enumerate(cores):
        out = np.einsum("ma,amb->mb", out, c[:, idx[:, k], :])
    return out[:, 0]


test_idx = np.random.default_rng(5).integers(0, 2, size=(4000, _n))
test_val = mystery(test_idx)
for rank in (2, 4, 8, 16):
    _calls["count"] = 0
    c = tn.tt_cross(mystery, [2] * _n, rank=rank, sweeps=3)
    n_calls = _calls["count"]
    err = np.linalg.norm(tt_eval(c, test_idx) - test_val) / np.linalg.norm(test_val)
    print(f"rank {rank:>2}: {n_calls:>7,d} evaluations "
          f"({100 * n_calls / 2**_n:>5.2f}% of the grid), rel err {err:.2e}")
print("\nThe error stops improving at rank 4: exp * cos is a sum of two complex")
print("exponentials (rank 2) and the extra sine adds 2 more. Four numbers per bond,")
print("found for the price of a few thousand evaluations instead of a quarter million.")

c = tn.tt_cross(mystery, [2] * _n, rank=4, sweeps=3)
plt.plot(np.linspace(0, 1, 2 ** _n, endpoint=False), tn.qtt_to_vector(c))
plt.title("mystery, reconstructed at rank 4")
plt.tight_layout()
plt.show()

## 9. TT-cross for encoding geometry: MPS masks

To place an immersed body (a cylinder, an airfoil) on a $2^n$ mesh without touching $2^n$
cells, encode an indicator $\theta(x)$ — one in the fluid, zero in the body — as an MPS and
apply it by an elementwise (Hadamard) product each step. A sharp *curved* boundary is
expensive in QTT; Peddinti *et al.* (2024) replace $\theta$ by a smooth mask with a knob
$\alpha$ and an error bar,

$$m(x) = 1 - e^{-\alpha (q(x) + |q(x)|)}, \qquad q|_{\partial\Omega} = 0,$$

which is exactly zero inside the body (where $q<0$) and rises smoothly outside it, and
build it with TT-cross from $q$ at a cost independent of the mesh. Below we use a dense
$512^2$ grid and TT-SVD only so that we can *measure* ranks and errors exactly.
(Below $q = 0.25^2 - |x - x_c|^2$ is positive inside the disc, so $m$ approximates the
disc indicator; for a solid disc in a flow, flip the sign of $q$.)

In [ ]:
m = 9
N = 2 ** m
xx = np.linspace(0, 1, N, endpoint=False)
X, Y = np.meshgrid(xx, xx, indexing="ij")
q = 0.25 ** 2 - ((X - 0.5) ** 2 + (Y - 0.5) ** 2)
indicator = (q > 0).astype(float)

print(f"{'alpha':>8} {'chi @ 1e-1':>11} {'chi @ 1e-2':>11} {'chi @ 1e-6':>11} "
      f"{'L2 to indicator':>17}")
print(f"{'sharp':>8} " + " ".join(
    f"{max(tn.tt_ranks(qq.from_grid(indicator, eps=e))):>11}"
    for e in (1e-1, 1e-2, 1e-6)) + f" {0.0:>17.4f}")
for a in (50, 200, 1000, 5000):
    mask = 1 - np.exp(-a * (q + np.abs(q)))
    chis = [max(tn.tt_ranks(qq.from_grid(mask, eps=e))) for e in (1e-1, 1e-2, 1e-6)]
    dist = np.linalg.norm(mask - indicator) / np.linalg.norm(indicator)
    print(f"{a:>8} " + " ".join(f"{c:>11}" for c in chis) + f" {dist:>17.4f}")

Two things to take from that table, and the second is the one people get wrong.

* The distance to the true indicator falls like $\alpha^{-1/2}$ (each fourfold increase
  in $\alpha$ roughly halves it): the bound $\sqrt{\ell_0^{D-2}/\alpha}$ from the paper,
  which makes the mask a *modelling choice with an error bar*.
* Smoothing does **not** reduce the rank at tight tolerance (`chi @ 1e-6`): the smooth mask
  is a different function with its own structure. The saving appears at the accuracy you
  actually want (`chi @ 1e-1`, `chi @ 1e-2`).

So $\alpha$ prices geometric fidelity against rank, and the price is known.

In [ ]:
mask = 1 - np.exp(-1000 * (q + np.abs(q)))
fig, ax = plt.subplots(1, 3, figsize=(10, 3))
ax[0].imshow(indicator.T, origin="lower", cmap="gray_r")
ax[0].set_title("sharp indicator")
ax[1].imshow(mask.T, origin="lower", cmap="gray_r")
ax[1].set_title(r"smoothed, $\alpha = 1000$")
ax[2].plot(xx, indicator[:, N // 2], label="sharp")
ax[2].plot(xx, mask[:, N // 2], label=r"$\alpha=1000$")
ax[2].set(xlim=(0.2, 0.35), title="the edge, close up")
ax[2].legend()
for a_ in ax[:2]:
    a_.set_xticks([]), a_.set_yticks([])
plt.tight_layout()
plt.show()

---
### Exercise 5 — price the geometry

$\alpha$ is a dial between fidelity and rank; set it from a requirement rather than by
taste. For the disc of radius $0.25$ on a $512^2$ grid:

1. Confirm that the relative $L_2$ distance between the mask and the true indicator
   scales like $\alpha^{-1/2}$ (fit a slope on log–log axes).
2. Requirement: "geometry accurate to 5%". Find the smallest $\alpha$ that meets it, and
   report the bond dimension needed to represent that mask to relative tolerance
   $10^{-2}$.
3. Compare with the sharp indicator at the same tolerance. Is the smoothing buying you
   anything here? Be honest about the answer.

In [ ]:
smooth_mask = lambda a: 1 - np.exp(-a * (q + np.abs(q)))

# TODO

#### Solution — Exercise 5

In [ ]:
alphas = np.array([25, 50, 100, 200, 400, 800, 1600, 3200])
dists = np.array([np.linalg.norm(smooth_mask(a) - indicator) / np.linalg.norm(indicator)
                  for a in alphas])
slope = np.polyfit(np.log(alphas), np.log(dists), 1)[0]
print(f"fitted exponent: {slope:.3f}   (theory: -0.5)")

a_star = alphas[dists < 0.05][0]
print(f"smallest tabulated alpha meeting 5%: {a_star} "
      f"(distance {dists[alphas == a_star][0]:.3f})")
chi_mask = max(tn.tt_ranks(qq.from_grid(smooth_mask(a_star), eps=1e-2)))
chi_sharp = max(tn.tt_ranks(qq.from_grid(indicator, eps=1e-2)))
print(f"bond dimension at 1e-2 tolerance: smoothed {chi_mask}, sharp {chi_sharp}")
print('''
At 5% geometric accuracy the smoothed mask is no cheaper than the indicator (here
slightly more expensive), and at tight tolerance it is more expensive too. The smoothing earns its keep at loose accuracy
and, more importantly, by being differentiable and by giving the approximation an error
bar. "alpha prices the trade between geometric fidelity and rank" is the defensible
statement; "smoothing reduces rank" is folklore.''')
plt.loglog(alphas, dists, "o-")
plt.loglog(alphas, dists[0] * (alphas / alphas[0]) ** -0.5, "k--", label=r"$\alpha^{-1/2}$")
plt.xlabel(r"$\alpha$"), plt.ylabel("relative $L_2$ distance to indicator"), plt.legend()
plt.tight_layout()
plt.show()

## 10. Open problems

* **TT preconditioning for stiff problems.** The condition number of $L$ grows like $4^n$;
  §5 showed what that does to the error at $n=14$. Multilevel preconditioners with
  provable rank bounds exist for elliptic problems (Bachmayr & Kazeev 2020), but there is
  no general recipe.
* **Explicit QTT construction from interpolation schemes** — instead of TT-cross, build
  the cores directly from multiscale interpolation (Lindsey 2023; Guzman, Tiunov & Aolita
  2026, upsampling).
* **Distributed TT algorithms.** Sweeps are sequential; GPU and multicore implementations
  (Hölscher *et al.* 2025; Röhrig-Zöllner *et al.* 2025) are only a start.
* **Robust TT recovery.** TT-cross overestimates ranks under noise (§8). Can this be
  solved?

That is the toolbox for Part III: an implicit solve (the pressure Poisson step) and a mask
built by TT-cross (the immersed body).